# Prepare CIFAR-100 TGSC crop banks

Enable a Kaggle GPU and Internet for cloning/installing/downloading. Attach CIFAR-100; teacher weights can be downloaded automatically below or attached. All training logic lives in the shared Python runner. FitNet is deferred.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Edit this cell. Paths below are examples; use your attached artifact locations.
REPO_URL = "https://github.com/ahmad903501/mdistiller-tgsc.git"
PRIVATE_REPO = True  # Add a Kaggle Secret named GITHUB_TOKEN with repository read access.
REVISION = "main"  # Prefer a fixed commit for reproducibility.
REPO_DIR = Path("/kaggle/working/mdistiller")
CONFIG_NAME = "dkd_resnet32x4_to_resnet8x4.json"
GPU = 0  # One GPU per process; other jobs can use another GPU.
SEED = 27
DATA_ROOT = Path("/kaggle/input/cifar100")  # Parent of cifar-100-python/
DOWNLOAD_TEACHER = True  # False to use attached weights at TEACHER_CHECKPOINT.
TEACHER_DOWNLOAD_ROOT = Path("/kaggle/working/teacher_weights")
TEACHER_CHECKPOINT = Path("/kaggle/input/cifar-teachers/resnet32x4_vanilla/ckpt_epoch_240.pth")
CACHE_ROOT = Path("/kaggle/input/tgsc-crop-bank")  # Parent of bank hash folders
CACHE_WRITE_ROOT = Path("/kaggle/working/tgsc_cache")
OUTPUT_DIR = Path("/kaggle/working/tgsc_runs/prepare_res32x4_res8x4_seed27")
RESUME = None  # E.g. Path('/kaggle/input/previous-run/latest.pt')
DOWNLOAD = False


In [ ]:
if not REPO_DIR.exists():
    if "YOUR_ACCOUNT" in REPO_URL:
        raise ValueError("Set REPO_URL to your pushed repository")
    def clone_repository():
        if not PRIVATE_REPO:
            subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
            return
        import os
        import tempfile
        from kaggle_secrets import UserSecretsClient
        # Keep the token out of URLs, notebook output, and files.
        clone_env = dict(os.environ, TGSC_GITHUB_TOKEN=UserSecretsClient().get_secret("GITHUB_TOKEN"),
                         GIT_TERMINAL_PROMPT="0")
        with tempfile.TemporaryDirectory() as auth_directory:
            askpass = Path(auth_directory) / "askpass.py"
            askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['TGSC_GITHUB_TOKEN'])\n")
            askpass.chmod(0o700)
            clone_env["GIT_ASKPASS"] = str(askpass)
            subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], env=clone_env, check=True)
    clone_repository()
    subprocess.run(["git", "checkout", REVISION], cwd=REPO_DIR, check=True)
else:
    wanted = subprocess.check_output(["git", "rev-parse", REVISION], cwd=REPO_DIR, text=True).strip()
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    if wanted != actual:
        raise RuntimeError("Existing checkout differs from REVISION; select the intended checkout before running")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-tgsc.txt")], check=True)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())


## Download and verify the official teacher

With `DOWNLOAD_TEACHER=True`, this cell downloads the official mdistiller CIFAR-100 archive (about 348 MiB), locates the teacher selected by `CONFIG_NAME`, extracts that checkpoint, and verifies its SHA-256 against the archive, strict state-dict loading, and a finite 100-class forward pass on CPU. Internet must be enabled. An existing archive is reused. Files go to writable `/kaggle/working/teacher_weights/`.

Set `DOWNLOAD_TEACHER=False` to keep using an attached checkpoint. No Google Drive file IDs or manual checkpoint renaming are needed. Preserve the teacher weights as a Kaggle Dataset for later training notebooks.


In [ ]:
if DOWNLOAD_TEACHER:
    subprocess.run([
        sys.executable, str(REPO_DIR / "tools" / "download_teachers.py"),
        "--config", str(REPO_DIR / "configs" / "tgsc" / "cifar100" / CONFIG_NAME),
        "--output-dir", str(TEACHER_DOWNLOAD_ROOT),
    ], cwd=REPO_DIR, check=True)
    # Read the actual model registry path; do not guess capitalization or directory names.
    import json
    from mdistiller.models import cifar_model_dict
    teacher_name = json.loads((REPO_DIR / "configs" / "tgsc" / "cifar100" / CONFIG_NAME).read_text())["teacher_model"]
    registered = Path(cifar_model_dict[teacher_name][1])
    TEACHER_CHECKPOINT = TEACHER_DOWNLOAD_ROOT / "cifar_teachers" / registered.parent.name / registered.name
if not TEACHER_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Teacher checkpoint does not exist: {TEACHER_CHECKPOINT}")
print("Teacher checkpoint:", TEACHER_CHECKPOINT)


In [ ]:
CONFIG = REPO_DIR / "configs" / "tgsc" / "cifar100" / CONFIG_NAME
RUNNER = REPO_DIR / "tools" / "run_tgsc.py"

def run_tgsc(command, *extra):
    args = [sys.executable, "-u", str(RUNNER), command,
            "--config", str(CONFIG), "--gpu", str(GPU), "--seed", str(SEED),
            "--data-root", str(DATA_ROOT), "--cache-root", str(CACHE_ROOT)]
    if command != "evaluate":
        args += ["--teacher-checkpoint", str(TEACHER_CHECKPOINT)]
    if DOWNLOAD:
        args += ["--download"]
    args += [str(value) for value in extra]
    log_dir = CACHE_WRITE_ROOT if command == "prepare" else OUTPUT_DIR
    log_dir.mkdir(parents=True, exist_ok=True)
    with (log_dir / (command + ".log")).open("a", encoding="utf-8") as log:
        process = subprocess.Popen(args, cwd=REPO_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, args)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise


## Prepare once per teacher

This writes base crops, raw r1/r2/r3, composite q and manifests for all curriculum sizes. Existing compatible files are reused. Different methods/students can attach the same saved teacher bank. Different teacher checkpoints need separate banks.


In [ ]:
run_tgsc("prepare", "--cache-write-root", CACHE_WRITE_ROOT)


## Preserve the cache

Save the notebook output or create a Kaggle Dataset from `tgsc_cache/`, then attach it to each method notebook. Set `CACHE_ROOT` there to the directory containing the bank hash folders. A directory in this session is not automatically shared with other notebooks.
